In [1]:
import os
import cv2
import numpy as np
import random
from tqdm import tqdm
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

In [2]:
def random_geometric(img):
    # flip
    if random.random() > 0.5:
        img = cv2.flip(img, 1)

    # small rotation
    angle = random.uniform(-20, 20)
    h, w = img.shape[:2]
    M = cv2.getRotationMatrix2D((w//2, h//2), angle, 1)
    img = cv2.warpAffine(img, M, (w, h))

    return img


def random_color(img):
    # brightness
    value = random.randint(-40, 40)
    img = cv2.add(img, value)

    # contrast
    alpha = random.uniform(0.7, 1.3)
    img = cv2.convertScaleAbs(img, alpha=alpha, beta=0)

    return img


def random_kernel(img):
    # randomly blur or sharpen
    if random.random() > 0.5:
        img = cv2.GaussianBlur(img, (5,5), 0)
    else:
        kernel = np.array([[0,-1,0],[-1,5,-1],[0,-1,0]])
        img = cv2.filter2D(img, -1, kernel)
    return img


def random_erasing(img):
    h, w = img.shape[:2]
    erase_w = random.randint(5, 20)
    erase_h = random.randint(5, 20)

    x = random.randint(0, w-erase_w)
    y = random.randint(0, h-erase_h)

    img[y:y+erase_h, x:x+erase_w] = random.randint(0,255)
    return img


def random_noise(img):
    noise = np.random.normal(0, 10, img.shape).astype(np.uint8)
    img = cv2.add(img, noise)
    return img


def full_augment(img):
    img = random_geometric(img)
    img = random_color(img)
    img = random_kernel(img)
    img = random_erasing(img)
    img = random_noise(img)
    return img

In [3]:
source_train = "train"
source_test = "test"

balanced_train = "balanced_train"
#balanced_test = "balanced_test"

os.makedirs(balanced_train, exist_ok=True)
#os.makedirs(balanced_test, exist_ok=True)

classes = [c for c in os.listdir(source_train) 
           if os.path.isdir(os.path.join(source_train, c))]

# Find max class size
class_counts = {c: len(os.listdir(os.path.join(source_train,c))) for c in classes}
max_count = max(class_counts.values())

print("Original class counts:", class_counts)
print("Target per class:", max_count)

Original class counts: {'happy': 7215, 'sad': 4830, 'fear': 4097, 'surprise': 3171, 'neutral': 4965, 'angry': 3995, 'disgust': 436}
Target per class: 7215


In [5]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

In [13]:
## DONT RUN AGAIN UNLESS balanced_train IS DELETED ##

import os, cv2, random, numpy as np
from tqdm import tqdm

source_train = "train"
balanced_train = "balanced_train"

os.makedirs(balanced_train, exist_ok=True)

# ignore mac junk
classes = [c for c in os.listdir(source_train) 
           if os.path.isdir(os.path.join(source_train,c))]

# original counts
class_counts = {
    c: len([f for f in os.listdir(os.path.join(source_train,c)) if not f.startswith('.')])
    for c in classes
}

max_count = max(class_counts.values())
print("Original counts:", class_counts)
print("Target:", max_count)

# -------- augmentation ----------
def augment(img):
    if random.random()>0.5:
        img = cv2.flip(img,1)

    angle = random.uniform(-25,25)
    h,w = img.shape[:2]
    M = cv2.getRotationMatrix2D((w//2,h//2), angle,1)
    img = cv2.warpAffine(img,M,(w,h))

    alpha = random.uniform(0.7,1.3)
    beta = random.randint(-30,30)
    img = cv2.convertScaleAbs(img, alpha=alpha, beta=beta)

    if random.random()>0.5:
        img = cv2.GaussianBlur(img,(5,5),0)
    else:
        kernel = np.array([[0,-1,0],[-1,5,-1],[0,-1,0]])
        img = cv2.filter2D(img,-1,kernel)

    return img

# -------- create balanced set ----------
for c in classes:
    os.makedirs(os.path.join(balanced_train,c), exist_ok=True)

    images = [f for f in os.listdir(os.path.join(source_train,c)) if not f.startswith('.')]
    current = len(images)

    # copy originals
    for img_name in images:
        src = os.path.join(source_train,c,img_name)
        dst = os.path.join(balanced_train,c,img_name)
        img = cv2.imread(src)
        cv2.imwrite(dst,img)

    needed = max_count - current

    for i in tqdm(range(needed), desc=f"Augmenting {c}"):
        img_name = random.choice(images)
        src = os.path.join(source_train,c,img_name)
        img = cv2.imread(src)

        aug = augment(img)
        new_name = f"aug_{i}_{img_name}"
        cv2.imwrite(os.path.join(balanced_train,c,new_name), aug)

print("Balanced dataset created correctly")

Original counts: {'happy': 7215, 'sad': 4830, 'fear': 4097, 'surprise': 3171, 'neutral': 4965, 'angry': 3995, 'disgust': 436}
Target: 7215


Augmenting happy: 0it [00:00, ?it/s]
Augmenting disgust: 100%|█████████████████| 6779/6779 [00:00<00:00, 7588.19it/s]

Balanced dataset created correctly


In [14]:
import os
for c in os.listdir("balanced_train"):
    if os.path.isdir(f"balanced_train/{c}"):
        print(c, len(os.listdir(f"balanced_train/{c}")))

happy 7215
sad 7215
fear 7215
surprise 7215
neutral 7215
angry 7215
disgust 7215
